### Using CNN-LSTM to classify violent clips

In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    # Jupyter notebook fallback
    BASE_DIR = Path.cwd().parent

# The video should be converted to frames first. During training, three frames are extracted per second,
# each frame is resized to 64x64 pixels, and the LSTM layer takes 10 consecutive frames as input.
def preprocess_video(video_path, frame_count=10, frame_size=(64, 64)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while len(frames) < frame_count:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, frame_size)
        frames.append(frame)
    cap.release()
    
    if len(frames) == 0:
        return np.zeros((frame_count, frame_size[0], frame_size[1], 3))
    elif len(frames) < frame_count:
        frames.extend([np.zeros_like(frames[0])]*(frame_count - len(frames)))
    return np.array(frames)

def load_test_data(folder_path, frame_count=10, frame_size=(64, 64)):
    test_data = []
    video_files = [f for f in os.listdir(folder_path) if f.endswith('.mp4') or f.endswith('.avi')]
    for video_file in video_files:
        video_path = os.path.join(folder_path, video_file)
        frames = preprocess_video(video_path, frame_count, frame_size)
        test_data.append(frames)
    return np.array(test_data), video_files

# Folder containing your test videos
test_video_folder = BASE_DIR / "data" / "streaming videos" / "filter_results"
# test_video_folder = BASE_DIR / "data" / "processed" / "violence-detection-dataset" / "high-level violence" / "cam2"
# test_video_folder = BASE_DIR / "data" / "raw" / "Real Life Violence Dataset" / "Violence"
test_data, test_video_files = load_test_data(test_video_folder)
test_data = test_data / 255.0

# This model can achieve over 90% accuracy on the training set and 77% on the validation set.
save_dir = BASE_DIR / "train_weight"
model_path = save_dir / "violence_detection_model_conv_64_lstm_64v2.h5"
model = tf.keras.models.load_model(model_path, compile=False)

# Recompile the model with a compatible optimizer and loss function
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

predictions = model.predict(test_data)
predicted_classes = np.argmax(predictions, axis=1)

class_names = ["high-level violence", "low-level violence", "non-violence"]
for video_file, predicted_class in zip(test_video_files, predicted_classes):
    print(f"Video: {video_file}, Predicted Class: {class_names[predicted_class]}")

2026-01-15 09:54:57.228772: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-15 09:54:57.284196: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-15 09:55:09.072277: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/mnt/d/Myworkplace/Python/violence-movies/.venv/lib/python3.12/site-

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 824ms/step
Video: Misson_ Impossible - Fallout_sample_clip_1_00_37_to_00_49.mp4, Predicted Class: high-level violence
Video: Misson_ Impossible - Fallout_sample_clip_2_00_51_to_01_18.mp4, Predicted Class: low-level violence
Video: Misson_ Impossible - Fallout_sample_clip_3_01_30_to_01_45.mp4, Predicted Class: high-level violence
Video: Misson_ Impossible - Fallout_sample_clip_4_01_47_to_02_03.mp4, Predicted Class: low-level violence
Video: Misson_ Impossible - Fallout_sample_clip_5_02_07_to_02_20.mp4, Predicted Class: high-level violence
Video: Misson_ Impossible - Fallout_sample_clip_6_02_38_to_04_17.mp4, Predicted Class: low-level violence
Video: Misson_ Impossible - Fallout_sample_clip_7_04_45_to_05_05.mp4, Predicted Class: high-level violence
Video: Misson_ Impossible - Fallout_sample_clip_8_05_07_to_05_14.mp4, Predicted Class: high-level violence
